In [142]:
import pandas as pd

In [143]:
import re
import ast
import pandas as pd
import numpy as np

def _parse_kinase_coord(x):
    """
    Accepts things like:
      - [[13, 271]]
      - "[[13, 271]]"
      - "13,271"  (fallback)
    Returns list of (start,end) tuples (1-indexed, inclusive).
    """
    if isinstance(x, list):
        coord = x
    elif isinstance(x, str):
        s = x.strip()
        try:
            coord = ast.literal_eval(s)
        except Exception:
            nums = list(map(int, re.findall(r"-?\d+", s)))
            if len(nums) >= 2:
                coord = [[nums[0], nums[1]]]
            else:
                coord = []
    else:
        coord = []

    pairs = []
    for item in coord:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            pairs.append((int(item[0]), int(item[1])))
    return pairs

def _extract_15mer_centered(seq: str, p_ind_1: int, p_aa: str, pad_char: str = ""):
    """
    15-aa window centered at p_ind_1 (1-indexed). Inclusive window [p-7, p+7].
    Pads with pad_char if near termini.
    Returns (window, ok_match) where ok_match indicates seq[p_ind_1-1]==p_aa (if possible).
    """
    if not isinstance(seq, str) or len(seq) == 0 or pd.isna(p_ind_1):
        return (np.nan, False)

    p = int(p_ind_1)
    if p < 1:
        return (np.nan, False)

    ok = True
    if isinstance(p_aa, str) and len(p_aa) == 1 and p <= len(seq):
        ok = (seq[p-1] == p_aa)

    left = p - 7
    right = p + 7

    chars = []
    for i in range(left, right + 1):
        if 1 <= i <= len(seq):
            chars.append(seq[i-1])
        else:
            chars.append(pad_char)
    return ("".join(chars), ok)

def _extract_domain(seq: str, coords, join_char: str = ""):
    """
    Extract kinase domain based on coords = list of (start,end) 1-indexed inclusive.
    If multiple segments, concatenates them in order (default join_char="").
    """
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    if not coords:
        return np.nan

    parts = []
    for (start, end) in coords:
        if start < 1 or end < 1:
            continue
        if end < start:
            start, end = end, start
        parts.append(seq[start-1:end])
    if not parts:
        return np.nan
    return join_char.join(parts)

def _collapse_conflicting_labels(
    df_train: pd.DataFrame,
    key_cols=("kinase_seq", "substrate_seq"),
    label_col="label",
):
    """
    If the same (kinase_seq, substrate_seq) appears with both label 0 and 1,
    keep only the label==1 rows for that pair.

    - Does NOT merge/average scores; it just filters.
    - If a pair appears only with label 0 -> keep (all) 0 rows
      (typically there should be only one row, but we don't assume).
    - If a pair appears only with label 1 -> keep (all) 1 rows.
    """
    # Normalize label to numeric {0,1}
    df = df_train.copy()
    df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
    df = df.dropna(subset=[label_col])
    df[label_col] = df[label_col].astype(int)

    # Identify keys that have any positive
    has_pos = (
        df.groupby(list(key_cols))[label_col]
          .max()
          .rename("_has_pos")
          .reset_index()
    )
    df = df.merge(has_pos, on=list(key_cols), how="left")

    # If a key has a positive, drop the negatives for that key
    df = df.loc[~((df["_has_pos"] == 1) & (df[label_col] == 0))].copy()
    df = df.drop(columns=["_has_pos"], errors="ignore").reset_index(drop=True)
    return df

def make_training_sequences(
    df_raw: pd.DataFrame,
    substrate_seq_col: str = "substrate_seq",
    kinase_seq_col: str = "kinase_seq",
    p_ind_col: str = "p_ind",
    p_aa_col: str = "p_aa",
    kinase_coord_col: str = "kinase_coord",
    label_col: str = "label",
    pad_char: str = "",
    drop_mismatch: bool = False,
    collapse_conflicting_labels: bool = True,
    kinase_domain_ready = False
):
    """
    Returns:
      df_out with new columns:
        - substrate_seq_15aa
        - kinase_domain_seq
        - p_aa_match (bool)
      plus a minimal df_train with columns:
        - kinase_seq (domain)
        - substrate_seq (15aa)
        - label

    Extra behavior:
      - If collapse_conflicting_labels=True, then for any (kinase_seq, substrate_seq)
        pair that has both label 0 and 1, only keep the label==1 rows.
    """
    df = df_raw.copy()

    # parse kinase coords
    df["_kinase_pairs"] = df[kinase_coord_col].apply(_parse_kinase_coord)

    # kinase domain seq
    if not kinase_domain_ready:
        df["kinase_domain_seq"] = [
            _extract_domain(seq, pairs)
            for seq, pairs in zip(df[kinase_seq_col], df["_kinase_pairs"])
        ]
    else:
        df["kinase_domain_seq"] = df['kinase_seq']
    # substrate 15mer
    out = [
        _extract_15mer_centered(seq, pind, paa, pad_char=pad_char)
        for seq, pind, paa in zip(df[substrate_seq_col], df[p_ind_col], df[p_aa_col])
    ]
    df["substrate_seq_15aa"] = [x[0] for x in out]
    df["p_aa_match"] = [x[1] for x in out]

    # optionally drop rows where p_aa doesn't match the sequence at p_ind
    if drop_mismatch:
        df = df[df["p_aa_match"] == True].copy()

    # minimal training df for your pipeline
    df_train = df[["kinase_domain_seq", "substrate_seq_15aa", label_col]].copy()
    df_train = df_train.rename(
        columns={
            "kinase_domain_seq": "kinase_seq",
            "substrate_seq_15aa": "substrate_seq",
            label_col: "label",
        }
    )

    # drop NAs created by bad coords / bad indices
    df_train = df_train.dropna(subset=["kinase_seq", "substrate_seq", "label"]).reset_index(drop=True)

    # NEW: remove conflicts (keep only label==1 if both 0 and 1 exist for same pair)
    if collapse_conflicting_labels:
        df_train = _collapse_conflicting_labels(
            df_train,
            key_cols=("kinase_seq", "substrate_seq"),
            label_col="label",
        )

    # cleanup helper
    df = df.drop(columns=["_kinase_pairs"], errors="ignore")

    return df, df_train


In [34]:
screening_dat = pd.read_pickle('/ksi_ml/251203_ksi_reduced_df_tp.pkl')
# negative might actually not be negative, so benchmark on positive novel hits 
screening_dat_pos = screening_dat.loc[screening_dat['true_positive'],:]
screening_dat_pos = screening_dat_pos.loc[:,['substrate','kinase']].drop_duplicates()

screening_dat_pos.loc[screening_dat_pos['kinase']=='JAK2_HUMAN_1','kinase'] = 'JAK2_HUMAN'
screening_dat_pos.loc[screening_dat_pos['kinase']=='KS6B1_HUMAN_active','kinase'] = 'KS6B1_HUMAN'
screening_dat_pos.loc[screening_dat_pos['kinase']=='KS6B1_HUMAN_wt','kinase'] = 'KS6B1_HUMAN'
screening_dat_pos.loc[screening_dat_pos['kinase']=='PLK1_HUMAN_active','kinase'] = 'PLK1_HUMAN'

kinome_dat = pd.read_pickle('/kinase_ml/data/human_kinome_seq_230301.pkl')
kinase_mapping_uniprot = pd.read_csv('/ksi_ml/data/kinome_data/kinase_mapping_uniprot.tsv',sep = '\t')
kinase_mapping_uniprot = kinase_mapping_uniprot.loc[:,['From','Entry Name']]
kinase_mapping_uniprot.columns = ['kinase','kinase_human_name']
kinome_dat = pd.merge(kinome_dat,kinase_mapping_uniprot)

screening_dat_pos = pd.merge(screening_dat_pos,kinome_dat,left_on = 'kinase',right_on = 'kinase_human_name',how = 'left')
screening_dat_pos = screening_dat_pos.drop('kinase_x',axis =1)
screening_dat_pos.columns = ['substrate_seq', 'kinase_gene', 'kinase_name', 'pk_group', 'pk_family',
       'pk_subfamily', 'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord',
       'kinase_human_name']

yeast_pairs = screening_dat_pos
yeast_pairs = yeast_pairs.loc[yeast_pairs['substrate_seq'].apply(lambda s: len(s)==15),:]
yeast_pairs['p_ind'] = 8
yeast_pairs['p_aa'] = yeast_pairs['substrate_seq'].apply(lambda s:s[7])
yeast_pairs['label'] = True
df_aug, df_train = make_training_sequences(yeast_pairs, 
                                                   drop_mismatch=False,
                                                   kinase_domain_ready = False)

In [38]:
screening_dat_pos.loc[screening_dat_pos['kinase_seq']=='',:]

,substrate_seq,kinase_gene,kinase_name,pk_group,pk_family,pk_subfamily,kinase,kinase_seq,kinseq_len,kinase_coord,kinase_human_name


In [102]:
pairs_aug_v2 = pd.read_csv('/ksi_ml/data/pairs_aug_v2.csv').iloc[:,1:]

In [103]:
pairs_test_neg = pd.concat([pd.read_csv('/ksi_ml/data/splits_v2/test.csv'),
                            pd.read_csv('/ksi_ml/data/splits_v2/test2.csv'),
                           pd.read_csv('/ksi_ml/data/splits_v2/val.csv')])
pairs_test_neg = pairs_test_neg.loc[pairs_test_neg['label']==0,:].drop_duplicates()

In [104]:
pairs_test_neg_aug_v2 = pd.merge(pairs_test_neg,pairs_aug_v2,left_on = ['kinase_seq','substrate_seq'],right_on = ['kinase_domain_seq','substrate_seq_15aa']).drop_duplicates(['kinase_domain_seq','substrate_seq_15aa'])

In [106]:
yeast_pairs_neg = pd.merge(df_aug.loc[:,['kinase_domain_seq']],pairs_test_neg_aug_v2,
                           left_on = 'kinase_domain_seq',
                               right_on = 'kinase_domain_seq')

In [107]:
yeast_pairs_neg = yeast_pairs_neg.drop_duplicates(['kinase_domain_seq','substrate_seq_15aa'])

In [109]:
yeast_pairs_neg = yeast_pairs_neg.drop(['kinase_seq_x','substrate_seq_x','label_x'],axis = 1)

In [110]:
yeast_pairs_neg.columns = ['kinase_domain_seq',
       'substrate', 'substrate_seq', 'kinase', 'kinase_seq', 'p_ind',
       'p_aa', 'kinase_coord', 'label', 'substrate_seq_15aa', 'p_aa_match']

In [111]:
yeast_pairs_neg = yeast_pairs_neg.drop_duplicates(['kinase_domain_seq','substrate_seq_15aa','kinase_seq'])

In [112]:
yeast_pairs_neg = yeast_pairs_neg.loc[:,['substrate', 'substrate_seq', 'kinase', 'kinase_seq', 'p_ind', 'p_aa',
       'kinase_coord', 'label', 'kinase_domain_seq', 'substrate_seq_15aa',
       'p_aa_match']]

In [113]:
yeast_pairs_pos_neg = pd.concat([df_aug,yeast_pairs_neg])

In [114]:
df_aug, df_train = make_training_sequences(yeast_pairs_pos_neg, 
                                                   drop_mismatch=False,
                                                   kinase_domain_ready = False)

In [115]:
df_aug

,substrate_seq,kinase_gene,kinase_name,pk_group,pk_family,pk_subfamily,kinase,kinase_seq,kinseq_len,kinase_coord,kinase_human_name,p_ind,p_aa,label,kinase_domain_seq,substrate_seq_15aa,p_aa_match,substrate
0,AAAAVEEYSCEFGSA,FES,Tyrosine-protein kinase Fes/Fps,TK,Fer,NaN,P07332,MGFSSELCSPQGHGVLQQMQEAELRLLEGMRKWMAQRVKSDREYAG...,822.0,"[[561, 822]]",FES_HUMAN,8,Y,1,LVLGEQIGRGNFGEVFSGRLRADNTLVAVKSCRETLPPDLKAKFLQ...,AAAAVEEYSCEFGSA,True,NaN
1,AADRDEVYTIPNSLK,FES,Tyrosine-protein kinase Fes/Fps,TK,Fer,NaN,P07332,MGFSSELCSPQGHGVLQQMQEAELRLLEGMRKWMAQRVKSDREYAG...,822.0,"[[561, 822]]",FES_HUMAN,8,Y,1,LVLGEQIGRGNFGEVFSGRLRADNTLVAVKSCRETLPPDLKAKFLQ...,AADRDEVYTIPNSLK,True,NaN
2,AADRDEVYTIPNSLK,FRK,Tyrosine-protein kinase FRK,TK,Src,NaN,P42685,MSNICQRLWEYLEPYLPCLSTEADKSTVIENPGALCSPQSQRHGHY...,505.0,"[[234, 491]]",FRK_HUMAN,8,Y,1,IQLLKRLGSGQFGEVWEGLWNNTTPVAVKTLKPGSMDPNDFLREAQ...,AADRDEVYTIPNSLK,True,NaN
3,AADRDEVYTIPNSLK,FYN,Tyrosine-protein kinase Fyn,TK,Src,NaN,P06241,MGCVQCKDKEATKLTEERDGSLNQSSGYRYGTDPTPQHYPSFGVTS...,537.0,"[[271, 524]]",FYN_HUMAN,8,Y,1,LQLIKRLGNGQFGEVWMGTWNGNTKVAIKTLKPGTMSPESFLEEAQ...,AADRDEVYTIPNSLK,True,NaN
4,AADRDEVYTIPNSLK,SYK,Tyrosine-protein kinase SYK,TK,Syk,NaN,P43405,MASSGMADSANHLPFFFGNITREEAEDYLVQGGMSDGLYLLRQSRN...,635.0,"[[371, 631]]",KSYK_HUMAN,8,Y,1,TLEDKELGSGNFGTVKKGYYQMKKVVKTVAVKILKNEANDPALKDE...,AADRDEVYTIPNSLK,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535467,MPGKRGLGWWWARLPLCLLLSLYGPWMPSSLGKPKGHPHMNSIRID...,NaN,NaN,NaN,NaN,NaN,P11309,MLLSKINSLAHLRAAPCNDLHATKLAPGKEKEPLESQYQVGPLLGS...,NaN,"[['38', '290']]",NaN,349,S,0,YQVGPLLGSGGFGSVYSGIRVSDNLPVAIKHVEKDRISDWGELPNG...,RGFDRYFSSRTLDNN,True,Q14833
535468,MDFQSCLYAIAEELGSEDLAALKFLCLDYIPHKKQETIEDAQKLFL...,NaN,NaN,NaN,NaN,NaN,P11309,MLLSKINSLAHLRAAPCNDLHATKLAPGKEKEPLESQYQVGPLLGS...,NaN,"[['38', '290']]",NaN,389,S,0,YQVGPLLGSGGFGSVYSGIRVSDNLPVAIKHVEKDRISDWGELPNG...,HTLEVDSSSHKNYIP,True,O89110
535469,MEVKGPSGRSFCCESEGQFKSCLKRHTPSLLLPSSWKGNSGSCLMA...,NaN,NaN,NaN,NaN,NaN,P11309,MLLSKINSLAHLRAAPCNDLHATKLAPGKEKEPLESQYQVGPLLGS...,NaN,"[['38', '290']]",NaN,274,S,0,YQVGPLLGSGGFGSVYSGIRVSDNLPVAIKHVEKDRISDWGELPNG...,QACEGGWSQGLPLPP,True,B1AL46
539076,MAAKTPSSEESGLPKLPVPPLQQTLATYLQCMRHLVSEEQFRKSQA...,NaN,NaN,NaN,NaN,NaN,Q8WZ42,MTTQAPTFTQPLQSVVVLEGSTATFEAHISGFPVPEVSWFRDGQVI...,NaN,"[['32178', '32432']]",NaN,346,S,0,YMIAEDLGRGEFGIVHRCVETSSKKTYMAKFVKVKGTDQVLVKKEI...,LLKHVTQSSRKLIRA,True,P28329-3


In [119]:
df_aug.to_csv('/ksi_ml/data/pairs_aug_ksi.csv')
df_train.to_csv('/ksi_ml/data/pairs_ksi.csv')

## Generate a more comprehensive dataset including also the non-tp in the KSI experiment

In [16]:
screening_dat = pd.read_pickle('/ksi_ml/251203_ksi_reduced_df_tp.pkl')
# negative might actually not be negative, so benchmark on positive novel hits 
screening_dat_pos = screening_dat.copy()
screening_dat_pos = screening_dat_pos.loc[:,['substrate','kinase']].drop_duplicates()

screening_dat_pos.loc[screening_dat_pos['kinase']=='JAK2_HUMAN_1','kinase'] = 'JAK2_HUMAN'
screening_dat_pos.loc[screening_dat_pos['kinase']=='KS6B1_HUMAN_active','kinase'] = 'KS6B1_HUMAN'
screening_dat_pos.loc[screening_dat_pos['kinase']=='KS6B1_HUMAN_wt','kinase'] = 'KS6B1_HUMAN'
screening_dat_pos.loc[screening_dat_pos['kinase']=='PLK1_HUMAN_active','kinase'] = 'PLK1_HUMAN'

kinome_dat = pd.read_pickle('/kinase_ml/data/human_kinome_seq_230301.pkl')
kinase_mapping_uniprot = pd.read_csv('/ksi_ml/data/kinome_data/kinase_mapping_uniprot.tsv',sep = '\t')
kinase_mapping_uniprot = kinase_mapping_uniprot.loc[:,['From','Entry Name']]
kinase_mapping_uniprot.columns = ['kinase','kinase_human_name']
kinome_dat = pd.merge(kinome_dat,kinase_mapping_uniprot)

screening_dat_pos = pd.merge(screening_dat_pos,kinome_dat,left_on = 'kinase',right_on = 'kinase_human_name',how = 'left')
screening_dat_pos = screening_dat_pos.drop('kinase_x',axis =1)
screening_dat_pos.columns = ['substrate_seq', 'kinase_gene', 'kinase_name', 'pk_group', 'pk_family',
       'pk_subfamily', 'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord',
       'kinase_human_name']

yeast_pairs = screening_dat_pos
yeast_pairs = yeast_pairs.loc[yeast_pairs['substrate_seq'].apply(lambda s: len(s)==15),:]
yeast_pairs['p_ind'] = 8
yeast_pairs['p_aa'] = yeast_pairs['substrate_seq'].apply(lambda s:s[7])
yeast_pairs['label'] = True
df_aug, df_train = make_training_sequences(yeast_pairs, 
                                                   drop_mismatch=False,
                                                   kinase_domain_ready = False)

In [17]:
pairs_aug_v2 = pd.read_csv('/ksi_ml/data/pairs_aug_v2.csv').iloc[:,1:]

In [18]:
pairs_test_neg = pd.concat([pd.read_csv('/ksi_ml/data/splits_v2/test.csv'),
                            pd.read_csv('/ksi_ml/data/splits_v2/test2.csv'),
                           pd.read_csv('/ksi_ml/data/splits_v2/val.csv')])
pairs_test_neg = pairs_test_neg.loc[pairs_test_neg['label']==0,:].drop_duplicates()

In [19]:
pairs_test_neg_aug_v2 = pd.merge(pairs_test_neg,pairs_aug_v2,left_on = ['kinase_seq','substrate_seq'],right_on = ['kinase_domain_seq','substrate_seq_15aa']).drop_duplicates(['kinase_domain_seq','substrate_seq_15aa'])

In [20]:
yeast_pairs_neg = pd.merge(df_aug.loc[:,['kinase_domain_seq']],pairs_test_neg_aug_v2,
                           left_on = 'kinase_domain_seq',
                               right_on = 'kinase_domain_seq')

In [21]:
yeast_pairs_neg = yeast_pairs_neg.drop_duplicates(['kinase_domain_seq','substrate_seq_15aa'])

In [22]:
yeast_pairs_neg = yeast_pairs_neg.drop(['kinase_seq_x','substrate_seq_x','label_x'],axis = 1)

In [23]:
yeast_pairs_neg.columns = ['kinase_domain_seq',
       'substrate', 'substrate_seq', 'kinase', 'kinase_seq', 'p_ind',
       'p_aa', 'kinase_coord', 'label', 'substrate_seq_15aa', 'p_aa_match']

In [24]:
yeast_pairs_neg = yeast_pairs_neg.drop_duplicates(['kinase_domain_seq','substrate_seq_15aa','kinase_seq'])

In [25]:
yeast_pairs_neg = yeast_pairs_neg.loc[:,['substrate', 'substrate_seq', 'kinase', 'kinase_seq', 'p_ind', 'p_aa',
       'kinase_coord', 'label', 'kinase_domain_seq', 'substrate_seq_15aa',
       'p_aa_match']]

In [26]:
yeast_pairs_pos_neg = pd.concat([df_aug,yeast_pairs_neg])

In [27]:
df_aug, df_train = make_training_sequences(yeast_pairs_pos_neg, 
                                                   drop_mismatch=False,
                                                   kinase_domain_ready = False)

In [29]:
df_aug.to_csv('/ksi_ml/data/pairs_aug_ksi_full.csv')
df_train.to_csv('/ksi_ml/data/pairs_ksi_full.csv')

## Generate KSI finetuning datasets

In [12]:
screening_dat = pd.read_csv('${DATA_MOUNT}/ksi_analysis_2025/260210_exp_ksi_fintuning_data.csv').iloc[:,1:]

screening_dat.loc[screening_dat['kinase']=='JAK2_HUMAN_1','kinase'] = 'JAK2_HUMAN'
screening_dat.loc[screening_dat['kinase']=='KS6B1_HUMAN_active','kinase'] = 'KS6B1_HUMAN'
screening_dat.loc[screening_dat['kinase']=='KS6B1_HUMAN_wt','kinase'] = 'KS6B1_HUMAN'
screening_dat.loc[screening_dat['kinase']=='PLK1_HUMAN_active','kinase'] = 'PLK1_HUMAN'

kinome_dat = pd.read_pickle('${DATA_MOUNT}/kinase_ml/data/human_kinome_seq_230301.pkl')
kinase_mapping_uniprot = pd.read_csv('/ksi_ml/data/kinome_data/kinase_mapping_uniprot.tsv',sep = '\t')
kinase_mapping_uniprot = kinase_mapping_uniprot.loc[:,['From','Entry Name']]
kinase_mapping_uniprot.columns = ['kinase','kinase_human_name']
kinome_dat = pd.merge(kinome_dat,kinase_mapping_uniprot)



In [13]:
screening_dat = pd.merge(screening_dat,kinome_dat,left_on = 'kinase',right_on = 'kinase_human_name',how = 'left')
screening_dat = screening_dat.drop('kinase_x',axis =1)

In [18]:
screening_dat.columns

Index(['substrate_seq_15aa', 'binding_max', 'fp_sub', 'true_positive', 'label',
       'kinase_gene', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily',
       'kinase_y', 'kinase_seq', 'kinseq_len', 'kinase_coord',
       'kinase_human_name'],
      dtype='object')

In [19]:
screening_dat.columns = ['substrate_seq', 'binding_max', 'fp_sub', 'true_positive', 'label',
       'kinase_gene', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily',
       'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord',
       'kinase_human_name']

In [21]:
screening_dat['substrate_seq'].apply(lambda s: len(s)==15).all()

np.True_

In [73]:
screening_dat = screening_dat.loc[screening_dat['substrate_seq'].apply(lambda s: len(s)==15),:]
screening_dat['p_ind'] = 8
screening_dat['p_aa'] = screening_dat['substrate_seq'].apply(lambda s:s[7])
df_aug, df_train = make_training_sequences(screening_dat, 
                                           drop_mismatch=False,
                                           kinase_domain_ready = False)

In [74]:
pairs_v2 = pd.read_csv('/ksi_ml/data/pairs_aug_v2.csv').iloc[:,1:]

In [75]:
# ── Find overlapping (kinase, substrate) pairs with conflicting labels ──
key_cols = ['kinase_domain_seq', 'substrate_seq_15aa']

ksi_in_v2 = pd.merge(
    df_aug[key_cols + ['label']],
    pairs_v2[key_cols + ['label']],
    on=key_cols,
).drop_duplicates()

# Pairs where the two datasets disagree on label (either direction → fix to 1)
conflict_mask = ksi_in_v2['label_x'] != ksi_in_v2['label_y']
conflict_pairs = ksi_in_v2.loc[conflict_mask, key_cols].drop_duplicates()

print(f"Found {len(conflict_pairs)} conflicting (kinase, substrate) pairs")
print(f"  screening=1, v2=0 (missed in literature):  "
      f"{((ksi_in_v2['label_x']==1)&(ksi_in_v2['label_y']==0)).sum()}")
print(f"  screening=0, v2=1 (FN in screening):       "
      f"{((ksi_in_v2['label_x']==0)&(ksi_in_v2['label_y']==1)).sum()}")

# ── Fix: set label → 1 in BOTH dataframes for all conflicting pairs ──
# Build a fast lookup set
conflict_set = set(zip(conflict_pairs['kinase_domain_seq'],
                       conflict_pairs['substrate_seq_15aa']))

mask_aug = pd.Series(
    list(zip(df_aug['kinase_domain_seq'], df_aug['substrate_seq_15aa'])),
    index=df_aug.index
).isin(conflict_set)

mask_v2 = pd.Series(
    list(zip(pairs_v2['kinase_domain_seq'], pairs_v2['substrate_seq_15aa'])),
    index=pairs_v2.index
).isin(conflict_set)

n_fix_aug = (df_aug.loc[mask_aug, 'label'] == 0).sum()
n_fix_v2  = (pairs_v2.loc[mask_v2, 'label'] == 0).sum()

df_aug.loc[mask_aug, 'label']  = 1
pairs_v2.loc[mask_v2, 'label'] = 1
df_train.loc[mask_aug, 'label']  = 1
print(f"Fixed {n_fix_aug} rows in df_aug (0→1)")
print(f"Fixed {n_fix_v2} rows in pairs_v2 (0→1)")

Found 168 conflicting (kinase, substrate) pairs
  screening=1, v2=0 (missed in literature):  59
  screening=0, v2=1 (FN in screening):       109
Fixed 109 rows in df_aug (0→1)
Fixed 130 rows in pairs_v2 (0→1)


In [ ]:
# v2 test should still be used for original data testing
# ksi data should be train val test splitted, holding the important withold pairs (reporter hits)

In [99]:
reporter_sets = pd.concat([df_aug.loc[(df_aug['kinase_gene']=='TBK1')&(df_aug['substrate_seq_15aa']=='QGPSQTSTPTDVTAI'),['kinase_domain_seq','substrate_seq_15aa']],
                 df_aug.loc[(df_aug['kinase_gene']=='IKKb')&(df_aug['substrate_seq_15aa']=='PHEEPSNSLMAIECR'),['kinase_domain_seq','substrate_seq_15aa']]])
reporter_sets.columns = ['kinase_seq','substrate_seq']

In [78]:
df_aug.to_csv('/ksi_ml/data/pairs_aug_phosphopca_finetuning.csv')
df_train.to_csv('/ksi_ml/data/pairs_phosphopca_finetuning.csv')

In [87]:
pairs_v2_fixed = pairs_v2.loc[:,['kinase_domain_seq','substrate_seq_15aa','label']]
pairs_v2_fixed.columns = ['kinase_seq','substrate_seq','label']

In [162]:
pairs_v2_fixed.to_csv('/ksi_ml/data/pairs_v2_fixed.csv')

In [89]:
pairs_v2_train =  pd.read_csv('/ksi_ml/data/splits_v2/train.csv').iloc[:,:-1]
pairs_v2_val =  pd.read_csv('/ksi_ml/data/splits_v2/val.csv').iloc[:,:-1]
pairs_v2_test =  pd.read_csv('/ksi_ml/data/splits_v2/test.csv').iloc[:,:-1]
pairs_v2_test2 =  pd.read_csv('/ksi_ml/data/splits_v2/test2.csv').iloc[:,:-1]

In [90]:
pairs_v2_train = pd.merge(pairs_v2_fixed,pairs_v2_train)
pairs_v2_val = pd.merge(pairs_v2_fixed,pairs_v2_val)
pairs_v2_test = pd.merge(pairs_v2_fixed,pairs_v2_test)
pairs_v2_test2 = pd.merge(pairs_v2_fixed,pairs_v2_test2)

In [92]:
mkdir /ksi_ml/data/splits_public/

In [93]:
pairs_v2_train.to_csv('/ksi_ml/data/splits_public/train.csv')
pairs_v2_val.to_csv('/ksi_ml/data/splits_public/val.csv')
pairs_v2_test.to_csv('/ksi_ml/data/splits_public/test.csv')
pairs_v2_test2.to_csv('/ksi_ml/data/splits_public/test2.csv')

In [ ]:
# python make_splits_v2_no_leak.py \
#   --expanded_csv ../../data/pairs_phosphopca_finetuning.csv \
#   --splits_v1_dir ../../data/splits_public \
#   --splits_v2_dir ../../data/splits_phosphopca_finetuning \
#   --key_cols kinase_seq,substrate_seq \
#   --dedup_before_split \
#   --seed 42


In [128]:
splits_phosphopca_finetuning_train = pd.read_csv('../../data/splits_phosphopca_finetuning/train.csv')

In [141]:
reporter_sets_ls

{('WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCLEIQIMRRLTHPNVVAARDVPEGMQNLAPNDLPLLAMEYCQGGDLRKYLNQFENCCGLREGAILTLLSDIASALRYLHENRIIHRDLKPENIVLQQGEQRLIHKIIDLGYAKELDQGSLCTSFVGTLQYLAPELLEQQKYTVTVDYWSFGTLAFECITGFRPFLPNWQPVQWHSKVRQKSEVDIVVSEDLNGTVKFSSSLPYPNNLNSVLAERLEKWLQLMLMWHPRQRGTDPTYGPNGCF',
  'PHEEPSNSLMAIECR'),
 ('WLLSDILGQGATANVFRGRHKKTGDLFAIKVFNNISFLRPVDVQMREFEVLKKLNHKNIVKLFAIEEETTTRHKVLIMEFCPCGSLYTVLEEPSNAYGLPESEFLIVLRDVVGGMNHLRENGIVHRDIKPGNIMRVIGEDGQSVYKLTDFGAARELEDDEQFVSLYGTEEYLHPDMYERAVLRKDHQKKYGATVDLWSIGVTFYHAATGSLPFRPFEGPRRNKEVMYKIITGKPSGAISGVQKAENGPIDWSGDMPVSCSLSRGLQVLLTPVLANILEADQEKCWGFDQFFAETSDILHRMV',
  'QGPSQTSTPTDVTAI')}

In [129]:
reporter_sets_ls = set(zip(reporter_sets['kinase_seq'],
                       reporter_sets['substrate_seq']))
mask_aug = pd.Series(
    list(zip(splits_phosphopca_finetuning_train['kinase_seq'], splits_phosphopca_finetuning_train['substrate_seq'])),
    index=splits_phosphopca_finetuning_train.index
).isin(reporter_sets_ls)

In [130]:
splits_phosphopca_finetuning_train_final = splits_phosphopca_finetuning_train.loc[~mask_aug,:]
splits_phosphopca_finetuning_train_removed = splits_phosphopca_finetuning_train.loc[mask_aug,:]

In [131]:
splits_phosphopca_finetuning_val = pd.read_csv('../../data/splits_phosphopca_finetuning/val.csv')

In [132]:
mask_aug = pd.Series(
    list(zip(splits_phosphopca_finetuning_val['kinase_seq'], splits_phosphopca_finetuning_val['substrate_seq'])),
    index=splits_phosphopca_finetuning_val.index
).isin(reporter_sets_ls)

In [133]:
splits_phosphopca_finetuning_val_removed = splits_phosphopca_finetuning_val.loc[mask_aug,:]
splits_phosphopca_finetuning_val_final = splits_phosphopca_finetuning_val.loc[~mask_aug,:]


In [135]:
splits_phosphopca_finetuning_test = pd.read_csv('../../data/splits_phosphopca_finetuning/test.csv')

In [136]:
splits_phosphopca_finetuning_test_final = pd.concat([splits_phosphopca_finetuning_test,
                                                    splits_phosphopca_finetuning_val_removed,
                                                    splits_phosphopca_finetuning_train_removed])

In [137]:
splits_phosphopca_finetuning_train_final.to_csv('../../data/splits_phosphopca_finetuning/train.csv')
splits_phosphopca_finetuning_val_final.to_csv('../../data/splits_phosphopca_finetuning/val.csv')
splits_phosphopca_finetuning_test_final.to_csv('../../data/splits_phosphopca_finetuning/test.csv')

In [138]:
splits_phosphopca_finetuning_test_final.shape

(1184, 3)

In [139]:
splits_phosphopca_finetuning_val_final.shape

(1215, 3)

In [140]:
splits_phosphopca_finetuning_train_final.shape

(9634, 3)

## Refine yeast-human dataset

In [171]:
pairs_v2 = pd.read_csv('/ksi_ml/data/pairs_aug_yeast.csv').iloc[:,1:]

In [148]:
# ── Find overlapping (kinase, substrate) pairs with conflicting labels ──
key_cols = ['kinase_domain_seq', 'substrate_seq_15aa']

ksi_in_v2 = pd.merge(
    df_aug[key_cols + ['label']],
    pairs_v2[key_cols + ['label']],
    on=key_cols,
).drop_duplicates()

# Pairs where the two datasets disagree on label (either direction → fix to 1)
conflict_mask = ksi_in_v2['label_x'] != ksi_in_v2['label_y']
conflict_pairs = ksi_in_v2.loc[conflict_mask, key_cols].drop_duplicates()

print(f"Found {len(conflict_pairs)} conflicting (kinase, substrate) pairs")
print(f"  screening=1, v2=0 (missed in literature):  "
      f"{((ksi_in_v2['label_x']==1)&(ksi_in_v2['label_y']==0)).sum()}")
print(f"  screening=0, v2=1 (FN in screening):       "
      f"{((ksi_in_v2['label_x']==0)&(ksi_in_v2['label_y']==1)).sum()}")

# ── Fix: set label → 1 in BOTH dataframes for all conflicting pairs ──
# Build a fast lookup set
conflict_set = set(zip(conflict_pairs['kinase_domain_seq'],
                       conflict_pairs['substrate_seq_15aa']))

mask_aug = pd.Series(
    list(zip(df_aug['kinase_domain_seq'], df_aug['substrate_seq_15aa'])),
    index=df_aug.index
).isin(conflict_set)

mask_v2 = pd.Series(
    list(zip(pairs_v2['kinase_domain_seq'], pairs_v2['substrate_seq_15aa'])),
    index=pairs_v2.index
).isin(conflict_set)

n_fix_aug = (df_aug.loc[mask_aug, 'label'] == 0).sum()
n_fix_v2  = (pairs_v2.loc[mask_v2, 'label'] == 0).sum()

df_aug.loc[mask_aug, 'label']  = 1
pairs_v2.loc[mask_v2, 'label'] = 1
df_train.loc[mask_aug, 'label']  = 1
print(f"Fixed {n_fix_aug} rows in df_aug (0→1)")
print(f"Fixed {n_fix_v2} rows in pairs_v2 (0→1)")

Found 9 conflicting (kinase, substrate) pairs
  screening=1, v2=0 (missed in literature):  9
  screening=0, v2=1 (FN in screening):       0
Fixed 0 rows in df_aug (0→1)
Fixed 9 rows in pairs_v2 (0→1)


In [170]:
pairs_v2.to_csv('/ksi_ml/data/pairs_aug_yeast_fixed.csv')